In [1]:
from src import *
from myutils import *
import gc

In [2]:
tree = [
    # 'noisy/di_5px',
    # 'noisy/spade_5px',

    '0noise/di_5px',
    '0noise/spade_5px',

    '0noise/di_3px',
    '0noise/spade_3px',
]

In [3]:
@parallelize()
def main(_folder):
    src = os.path.join('__estimates__.old', _folder)
    dst = os.path.join('__estimates__', _folder)
    for file in tqdm(glob(src+'/*')):
        c = LoadEstimates(file)
        try:
            c_ = FrequencyEstimation.FromEstimates(c)
            if not os.path.exists(dst):
                os.makedirs(dst)
            c_.savez(dst)
        except (ValueError, RuntimeError):
            print(f'{file} error')

        gc.collect()

In [4]:
main(tree)

(None, None, None, None)

In [ ]:
c = LoadEstimates('__estimates__/noisy/di_5px/di_5px_f0.2_d98.npz')

In [ ]:
plot(c.cropped_data[0, 0])

In [ ]:
td_estimator('DI', c.cropped_data, c.noise_weight, standardize=False)

In [ ]:
c.noise / c.cropped_data.mean(-1)

In [ ]:
np.sign(0)

In [2]:
lst = np.arange(0, 101, 2)
lst

array([  0,   2,   4,   6,   8,  10,  12,  14,  16,  18,  20,  22,  24,
        26,  28,  30,  32,  34,  36,  38,  40,  42,  44,  46,  48,  50,
        52,  54,  56,  58,  60,  62,  64,  66,  68,  70,  72,  74,  76,
        78,  80,  82,  84,  86,  88,  90,  92,  94,  96,  98, 100])

In [3]:
for m in ('di', 'spade'):

    root = lambda x: os.path.join(f'__raw__/{m}_5px', x)

    for d in tqdm(lst):
        # f = np.round(_f, 5)

        raw = np.load(root(f'{m}_5px_f0.2_d{d}_raw.npy'))
        timestamp = np.load(root(f'{m}_5px_f0.2_d{d}_timestamp.npy'))

        meta = MetaData(m.upper(), 0.2, 5*DMD.PIXEL_SIZE/2, d, timestamp)

        try:
            c = FrequencyEstimation.FromRaw(raw, meta)

            if not os.path.exists(f'__estimates__/{today}/{m}_5px'):
                os.makedirs(f'__estimates__/{today}/{m}_5px')

            c.savez(rf'__estimates__/{today}/{m}_5px')
            gc.collect()
        except:
            print(f'{m}_d{d} not converged')


  0%|          | 0/51 [00:00<?, ?it/s]

100%|██████████| 51/51 [03:06<00:00,  3.66s/it]


In [ ]:
c = LoadEstimates(r'__estimates__\di_5px_5\di_5px_f0.4_d0.npz')

c_ = Estimates(c.cropped_data, c.metadata, noise=c.noise)

In [ ]:
# c__ = FrequencyEstimation.FromEstimates(c_)

In [ ]:
pn_f_mean = []

for f in lst:

    c = LoadEstimates(rf'__estimates__\di_5px_5\di_5px_f{np.round(f, 5)}_d0.npz')
    pn_f = []

    for i in range(200):
        pn_f.append(freq_estimator(c.time_domain[i], method='fft'))

    pn_f_mean.append(np.mean(pn_f))

In [ ]:
plot(lst, pn_f_mean)
plot(lst, lst)

In [ ]:
c = LoadEstimates(rf'__estimates__\di_5px_5\di_5px_f0.135_d0.npz')
pn_f = []

pn_f.append(freq_estimator(c.time_domain.ravel(), method='fft'))
print(np.mean(pn_f))

In [ ]:
len(c.time_domain.ravel())

In [ ]:
plot(np.linspace(0, 1, 50), abs_fft(c.time_domain[0]))
plot([0.14], [0], 'o')

In [ ]:
0.138 in np.linspace(0, 1, 10000)

In [ ]:

c = LoadEstimates(rf'__estimates__\di_5px_5\di_5px_f0.36_d0.npz')
for i in range(200):
    def waveform(f):
        n = np.arange(50)
        return np.sin(tau * f * n)

    def lsloss(f):
        
        return np.array([np.sum((c.time_domain[i] - waveform(_f))**2) for _f in f])

    plot(np.linspace(0.1, 0.4, 1000), lsloss(np.linspace(0.1, 0.4, 1000)))
    plot([c.frequency_estimates[i]], [0], 'o', xlim=(0.35, 0.4))